In [1]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [5]:
import zipfile
from pathlib import Path

drive_dir = Path("/content/drive/MyDrive/Colab Notebooks")
project_dir = drive_dir / "nlp_translation"

if not (project_dir / "config.json").exists():
    with zipfile.ZipFile(
        drive_dir / "nlp_translation_base_commune.zip", "r"
    ) as archive:
        archive.extractall(drive_dir)

print("Dossier prêt :", project_dir.exists())

Dossier prêt : True


In [6]:
from pathlib import Path
import sys

project_dir = Path("/content/drive/MyDrive/Colab Notebooks/nlp_translation")

if str(project_dir) not in sys.path:
    sys.path.insert(0, str(project_dir))

from src.data import load_prepared, make_loaders

config, vocab_fr, vocab_en, prepared = load_prepared(project_dir)

# Réduire la mémoire utilisée
config["batch_size"] = 8

loaders = make_loaders(prepared, config)

train_loader = loaders["train"]
validation_loader = loaders["validation"]
test_loader = loaders["test"]

print("Entraînement :", len(prepared["train"]))
print("Validation :", len(prepared["validation"]))
print("Test :", len(prepared["test"]))
print("Batch size :", config["batch_size"])

Entraînement : 30000
Validation : 2000
Test : 2000
Batch size : 8


In [ ]:
from pathlib import Path
import sys

project_dir = Path("/content/nlp_translation")

# Vérifier que le dossier est disponible
assert (project_dir / "config.json").exists(), (
    "Réimporte et décompresse le ZIP dans cette session Colab."
)

if str(project_dir) not in sys.path:
    sys.path.insert(0, str(project_dir))

from src.data import load_prepared, make_loaders

# Charger notamment la variable config
config, vocab_fr, vocab_en, prepared = load_prepared(project_dir)

loaders = make_loaders(prepared, config)
train_loader = loaders["train"]
validation_loader = loaders["validation"]
test_loader = loaders["test"]

print("Configuration chargée. Seed :", config["seed"])

Configuration chargée. Seed : 42


# **1. Imports et choix du GPU**

In [7]:
import random
import torch
import torch.nn as nn

from torch.nn.utils.rnn import pack_padded_sequence

PAD_ID = config["special_tokens"]["<PAD>"]

random.seed(config["seed"])
torch.manual_seed(config["seed"])

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Appareil utilisé :", device)

Appareil utilisé : cuda


# **2. Construire l’encodeur, le décodeur et le modèle complet**

In [8]:
# -------------------------------------
# A. Encodeur : lire la phrase française
# -------------------------------------
class EncoderLSTM(nn.Module):

    def __init__(self, vocab_size, embedding_dim, hidden_dim,
                 num_layers=1):
        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            embedding_dim,
            padding_idx=PAD_ID
        )

        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True
        )

    def forward(self, src, src_lengths):

        # Transformer les indices en vecteurs
        embedded = self.embedding(src)

        # Éviter que le LSTM lise les positions de padding
        packed = pack_padded_sequence(
            embedded,
            src_lengths.cpu(),
            batch_first=True,
            enforce_sorted=False
        )

        # Récupérer les états finaux de chaque phrase
        _, (hidden, cell) = self.lstm(packed)

        return hidden, cell


# -------------------------------------
# B. Décodeur : prédire un token anglais
# -------------------------------------
class DecoderLSTM(nn.Module):

    def __init__(self, vocab_size, embedding_dim, hidden_dim,
                 num_layers=1):
        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            embedding_dim,
            padding_idx=PAD_ID
        )

        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True
        )

        self.output_layer = nn.Linear(
            hidden_dim,
            vocab_size
        )

    def forward(self, token, hidden, cell):

        # token : un indice par phrase du batch
        embedded = self.embedding(token.unsqueeze(1))

        output, (hidden, cell) = self.lstm(
            embedded,
            (hidden, cell)
        )

        # Un score pour chaque token du vocabulaire anglais
        logits = self.output_layer(output.squeeze(1))

        return logits, hidden, cell


# -------------------------------------
# C. Relier l'encodeur et le décodeur
# -------------------------------------
class Seq2SeqLSTM(nn.Module):

    def __init__(self, encoder, decoder):
        super().__init__()

        self.encoder = encoder
        self.decoder = decoder

    def forward(self, src, src_lengths, decoder_input,
                teacher_forcing_ratio=0.5):

        # Résumé fixe transmis par l'encodeur
        hidden, cell = self.encoder(src, src_lengths)

        # Le premier token est <SOS>
        token = decoder_input[:, 0]

        steps = decoder_input.size(1)
        predictions = []

        for t in range(steps):

            logits, hidden, cell = self.decoder(
                token, hidden, cell
            )

            predictions.append(logits)

            if t + 1 < steps:

                use_teacher_forcing = (
                    random.random() < teacher_forcing_ratio
                )

                if use_teacher_forcing:
                    # Donner le vrai token précédent
                    token = decoder_input[:, t + 1]
                else:
                    # Donner le token prédit par le modèle
                    token = logits.argmax(dim=-1)

        # Forme : [batch, longueur cible, vocabulaire anglais]
        return torch.stack(predictions, dim=1)


# -------------------------------------
# D. Créer le modèle
# -------------------------------------
encoder = EncoderLSTM(
    vocab_size=len(vocab_fr),
    embedding_dim=config["embedding_dim"],
    hidden_dim=config["hidden_dim"],
    num_layers=config["num_layers"]
)

decoder = DecoderLSTM(
    vocab_size=len(vocab_en),
    embedding_dim=config["embedding_dim"],
    hidden_dim=config["hidden_dim"],
    num_layers=config["num_layers"]
)

model = Seq2SeqLSTM(encoder, decoder).to(device)

num_parameters = sum(
    p.numel() for p in model.parameters()
    if p.requires_grad
)

print("Nombre de paramètres entraînables :", num_parameters)

Nombre de paramètres entraînables : 5920528


# **Verification du modele**

In [9]:
batch = next(iter(train_loader))

src = batch["src"][:2].to(device)
src_lengths = batch["src_lengths"][:2]

decoder_input = batch["decoder_input"][:2, :5].to(device)
target = batch["target"][:2, :5].to(device)

criterion = nn.CrossEntropyLoss(ignore_index=PAD_ID)

model.eval()

with torch.no_grad():

    logits = model(
        src,
        src_lengths,
        decoder_input,
        teacher_forcing_ratio=1.0
    )

    loss = criterion(
        logits.reshape(-1, len(vocab_en)),
        target.reshape(-1)
    )

expected_shape = (
    src.size(0),
    decoder_input.size(1),
    len(vocab_en)
)

assert tuple(logits.shape) == expected_shape
assert torch.isfinite(loss).item()

print("Dimensions des sorties :", tuple(logits.shape))
print("Loss initiale :", loss.item())
print("Modèle vérifié : prêt pour l'entraînement.")

Dimensions des sorties : (2, 5, 10000)
Loss initiale : 9.19169807434082
Modèle vérifié : prêt pour l'entraînement.


# **Préparer l’entraînement**

In [10]:
import time
import json
import random
from pathlib import Path
from tqdm.auto import tqdm

random.seed(config["seed"])
torch.manual_seed(config["seed"])

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(config["seed"])

encoder = EncoderLSTM(
    len(vocab_fr),
    config["embedding_dim"],
    config["hidden_dim"],
    config["num_layers"]
)

decoder = DecoderLSTM(
    len(vocab_en),
    config["embedding_dim"],
    config["hidden_dim"],
    config["num_layers"]
)

model = Seq2SeqLSTM(encoder, decoder).to(device)

# Repartir du même ordre de batchs
loaders = make_loaders(prepared, config)
train_loader = loaders["train"]
validation_loader = loaders["validation"]
test_loader = loaders["test"]

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=config["learning_rate"]
)

# Somme des erreurs, sans compter le padding
criterion = nn.CrossEntropyLoss(
    ignore_index=PAD_ID,
    reduction="sum"
)

checkpoint_dir = project_dir / "checkpoints"
results_dir = project_dir / "results"

checkpoint_dir.mkdir(parents=True, exist_ok=True)
results_dir.mkdir(parents=True, exist_ok=True)

print("Appareil :", device)
print("Nombre d'époques :", config["epochs"])
print("Sauvegardes :", checkpoint_dir)

Appareil : cuda
Nombre d'époques : 5
Sauvegardes : /content/drive/MyDrive/Colab Notebooks/nlp_translation/checkpoints


# **Définir l’entraînement et la validation**

In [11]:
def train_one_epoch(model, loader):

    model.train()

    total_loss = 0.0
    total_tokens = 0

    for batch in tqdm(loader, desc="Entraînement"):

        src = batch["src"].to(device)
        lengths = batch["src_lengths"]  # Rester sur CPU
        decoder_input = batch["decoder_input"].to(device)
        target = batch["target"].to(device)

        optimizer.zero_grad()

        logits = model(
            src,
            lengths,
            decoder_input,
            teacher_forcing_ratio=config["teacher_forcing_ratio"]
        )

        loss_sum = criterion(
            logits.reshape(-1, len(vocab_en)),
            target.reshape(-1)
        )

        valid_tokens = target.ne(PAD_ID).sum().item()
        loss = loss_sum / valid_tokens

        if not torch.isfinite(loss).item():
            raise RuntimeError("Loss non finie : arrêter et vérifier le modèle.")

        loss.backward()

        # Limiter les gradients trop grands
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()

        total_loss += loss_sum.detach().item()
        total_tokens += valid_tokens

    return total_loss / total_tokens


@torch.no_grad()
def validate_one_epoch(model, loader):

    model.eval()

    total_loss = 0.0
    total_tokens = 0

    for batch in tqdm(loader, desc="Validation"):

        src = batch["src"].to(device)
        lengths = batch["src_lengths"]
        decoder_input = batch["decoder_input"].to(device)
        target = batch["target"].to(device)

        logits = model(
            src,
            lengths,
            decoder_input,
            teacher_forcing_ratio=1.0
        )

        loss_sum = criterion(
            logits.reshape(-1, len(vocab_en)),
            target.reshape(-1)
        )

        total_loss += loss_sum.item()
        total_tokens += target.ne(PAD_ID).sum().item()

    return total_loss / total_tokens

# **Lancer l’entraînement et sauvegarder**

In [12]:
history = []
best_validation_loss = float("inf")
total_training_seconds = 0.0

num_parameters = sum(
    p.numel() for p in model.parameters()
    if p.requires_grad
)

for epoch in range(1, config["epochs"] + 1):

    print(f"\nÉpoque {epoch}/{config['epochs']}")

    # Synchroniser le GPU pour mesurer le temps
    if device.type == "cuda":
        torch.cuda.synchronize()

    start = time.perf_counter()

    train_loss = train_one_epoch(model, train_loader)

    if device.type == "cuda":
        torch.cuda.synchronize()

    training_seconds = time.perf_counter() - start
    total_training_seconds += training_seconds

    validation_loss = validate_one_epoch(
        model,
        validation_loader
    )

    row = {
        "epoch": epoch,
        "train_loss": train_loss,
        "validation_loss": validation_loss,
        "training_seconds": training_seconds,
        "total_training_seconds": total_training_seconds
    }

    history.append(row)

    print(
        f"Train loss : {train_loss:.4f} | "
        f"Validation loss : {validation_loss:.4f} | "
        f"Temps entraînement : {training_seconds / 60:.2f} min"
    )

    improved = validation_loss < best_validation_loss

    if improved:
        best_validation_loss = validation_loss

    checkpoint = {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "config": config,
        "vocab_fr": vocab_fr,
        "vocab_en": vocab_en,
        "validation_loss": validation_loss,
        "best_validation_loss": best_validation_loss,
        "num_parameters": num_parameters,
        "history": history,
        "python_rng_state": random.getstate(),
        "torch_rng_state": torch.get_rng_state(),
        "cuda_rng_states": (
            torch.cuda.get_rng_state_all()
            if torch.cuda.is_available() else []
        )
    }

    # Sauvegarde de la dernière époque
    torch.save(
        checkpoint,
        checkpoint_dir / "lstm_last.pt"
    )

    # Sauvegarde du meilleur modèle
    if improved:
        torch.save(
            checkpoint,
            checkpoint_dir / "lstm_best.pt"
        )
        print("Meilleur modèle sauvegardé.")

    # Sauvegarde des résultats après chaque époque
    (results_dir / "lstm_history.json").write_text(
        json.dumps(history, indent=2),
        encoding="utf-8"
    )

print("\nEntraînement terminé.")
print("Meilleure loss de validation :", best_validation_loss)
print("Temps total d'entraînement :", total_training_seconds / 60, "min")


Époque 1/5


Entraînement:   0%|          | 0/3750 [00:00<?, ?it/s]

Validation:   0%|          | 0/250 [00:00<?, ?it/s]

Train loss : 5.8465 | Validation loss : 5.2678 | Temps entraînement : 4.45 min
Meilleur modèle sauvegardé.

Époque 2/5


Entraînement:   0%|          | 0/3750 [00:00<?, ?it/s]

...

Validation:   0%|          | 0/250 [00:00<?, ?it/s]

Train loss : 4.8867 | Validation loss : 4.8253 | Temps entraînement : 4.56 min
Meilleur modèle sauvegardé.

Époque 5/5


Entraînement:   0%|          | 0/3750 [00:00<?, ?it/s]

Validation:   0%|          | 0/250 [00:00<?, ?it/s]

Train loss : 4.7141 | Validation loss : 4.8041 | Temps entraînement : 4.56 min
Meilleur modèle sauvegardé.

Entraînement terminé.
Meilleure loss de validation : 4.80406633484728
Temps total d'entraînement : 22.57576052181666 min


# **Charge le meilleur modèle et calculer la loss de test**

In [13]:
checkpoint = torch.load(
    checkpoint_dir / "lstm_best.pt",
    map_location=device,
    weights_only=True
)

model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

test_loss = validate_one_epoch(model, test_loader)

print("Époque sélectionnée :", checkpoint["epoch"])
print("Loss de test :", round(test_loss, 4))

Validation:   0%|          | 0/250 [00:00<?, ?it/s]

Époque sélectionnée : 5
Loss de test : 4.792


# **Génére les traductions sans teacher forcing**

In [14]:
SOS_ID = config["special_tokens"]["<SOS>"]
EOS_ID = config["special_tokens"]["<EOS>"]
PAD_ID = config["special_tokens"]["<PAD>"]

id_to_token_en = {
    index: token for token, index in vocab_en.items()
}

# Limite de sécurité identique pour tous vos modèles
MAX_GENERATION_STEPS = 100


@torch.no_grad()
def translate_batch(src, src_lengths):

    model.eval()

    hidden, cell = model.encoder(src, src_lengths)

    batch_size = src.size(0)

    token = torch.full(
        (batch_size,),
        SOS_ID,
        dtype=torch.long,
        device=device
    )

    finished = torch.zeros(
        batch_size,
        dtype=torch.bool,
        device=device
    )

    generated_steps = []

    for _ in range(MAX_GENERATION_STEPS):

        logits, hidden, cell = model.decoder(
            token, hidden, cell
        )

        token = logits.argmax(dim=-1)

        # Les phrases déjà terminées restent terminées
        token = torch.where(
            finished,
            torch.full_like(token, EOS_ID),
            token
        )

        generated_steps.append(token.cpu())

        finished = finished | token.eq(EOS_ID)

        if finished.all().item():
            break

    generated_ids = torch.stack(
        generated_steps, dim=1
    ).tolist()

    translations = []

    for sequence in generated_ids:

        words = []

        for index in sequence:

            if index == EOS_ID:
                break

            if index not in {PAD_ID, SOS_ID}:
                words.append(
                    id_to_token_en.get(index, "<UNK>")
                )

        translations.append(" ".join(words))

    return translations, finished.cpu().tolist()

# **Traduire les 2 000 phrases de test**

In [15]:
import pandas as pd

evaluation_rows = []

for batch in tqdm(test_loader, desc="Traduction du test"):

    predictions, ended = translate_batch(
        batch["src"].to(device),
        batch["src_lengths"]
    )

    for source, reference, prediction, length, has_eos in zip(
        batch["source_texts"],
        batch["reference_texts"],
        predictions,
        batch["src_token_counts"].tolist(),
        ended
    ):

        evaluation_rows.append({
            "source": source,
            "reference": reference,
            "prediction": prediction,
            "source_length": length,
            "ended_with_eos": has_eos
        })

translations_df = pd.DataFrame(evaluation_rows)

assert len(translations_df) == 2000

translations_df.to_csv(
    results_dir / "lstm_test_translations.csv",
    index=False,
    encoding="utf-8"
)

display(
    translations_df[
        ["source", "reference", "prediction"]
    ].head(10)
)

print(
    "Phrases ayant atteint la limite sans <EOS> :",
    (~translations_df["ended_with_eos"]).sum()
)

Traduction du test:   0%|          | 0/250 [00:00<?, ?it/s]

,source,reference,prediction
0,- vous étiez en train de vous embrasser à l'ar...,- you were at a bus stop kissing him!,- you you you to you you you !
1,"avec une ironie farceuse, le jeune artiste tch...",with irony and mischief the young czech artist...,in the <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UN...
2,qui va parler au vendeur de voitures ? big fre...,- who's going to talk to the used car salesman?,"we have to <UNK> <UNK> , <UNK> <UNK> ."
3,on vous dit éclairé : votre musique parle d'hu...,"people think you are a great man, that your mu...","you know , you , you , you , and you , and you ."
4,autres coraux,lace corals,<UNK> <UNK>
5,"dites, je me demandais...","yeah, there's one man i know who could nail him.","oh , i . . . . ."
6,peter et moi étions dans un accident de voitur...,peter and i were in a car crash.,and i was a to me . . . . . .
7,ces lois mettent en place une république parle...,these laws establish a parliamentary republic ...,these are are the to of the and and and and the .
8,"les activités de suivi, de recherche et d'éval...","monitoring, research and evaluation activities...",the and of and and and and and and and and and...
9,lorsque les actes des institutions adoptés ava...,where acts of the institutions prior to 1 may ...,when the of of the the of the the the the of t...


Phrases ayant atteint la limite sans <EOS> : 22


# **Calculer le BLEU et sauvegarder les résultats**

In [17]:
%pip install -q sacrebleu

In [18]:
from sacrebleu.metrics import BLEU

# Comparaison insensible aux majuscules :
# votre préparation utilise des phrases en minuscules.
bleu_metric = BLEU(
    lowercase=True,
    tokenize="13a"
)

bleu_result = bleu_metric.corpus_score(
    translations_df["prediction"].tolist(),
    [translations_df["reference"].tolist()]
)

metrics = {
    "model": "LSTM sans attention",
    "best_epoch": checkpoint["epoch"],
    "validation_loss": checkpoint["validation_loss"],
    "test_loss": test_loss,
    "bleu": bleu_result.score,
    "bleu_signature": str(bleu_metric.get_signature()),
    "training_minutes": (
        checkpoint["history"][-1]["total_training_seconds"] / 60
    ),
    "num_parameters": checkpoint["num_parameters"],
    "max_generation_steps": MAX_GENERATION_STEPS,
    "test_sentences": len(translations_df),
    "sentences_without_eos": int(
        (~translations_df["ended_with_eos"]).sum()
    )
}

(results_dir / "lstm_metrics.json").write_text(
    json.dumps(metrics, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

display(pd.DataFrame([metrics]))

,model,best_epoch,validation_loss,test_loss,bleu,bleu_signature,training_minutes,num_parameters,max_generation_steps,test_sentences,sentences_without_eos
0,LSTM sans attention,5,4.804066,4.792028,1.240395,nrefs:1|case:lc|eff:no|tok:13a|smooth:exp|vers...,22.575761,5920528,100,2000,22
